# study what cell represent
- compilation of things needed to study what filtered cells represent (what precip fraction and anvil area)
- first execute nbooks 55 to 57 to calculate domain area-weighted rain amount/ month (to compare to cells)
- input: linking/filtered & aux/{data}_monthly_rain_amount
- output: None (printed)

In [1]:
import warnings

# ignore general categories
for cat in [FutureWarning, RuntimeWarning, UserWarning]:
    warnings.filterwarnings("ignore", category=cat)

# ignore specific repeated messages
for msg in [
    "pyproj unable to set PROJ database path",
    "As of v1.6.0, segmentation with time length 1",
    "Using DEFAULT_SPHERICAL_EARTH_RADIUS"
]:
    warnings.filterwarnings("ignore", message=msg)


import numpy as np
import xarray as xr
import os
import pandas as pd

from pathlib import Path

# Import tobac itself:


import sys


# import use_tobac

sys.path.append("../tools")
import plots_mapping as pm
import handle_OBS_data as ob
import handle_ICON_data as ic 
from defs_application import bad_timesteps

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
inputdir = "../output/linking/filtered"
models = ["ICON","OBS"]
timeperiods = ["JUNE","DECEMBER"]


In [3]:
## load tracks
ds = {}
for model in models: 
    ds[model] = {}
    for timeperiod in timeperiods: 
        file = os.path.join(inputdir,f"all_tracks_filtered_{model}_{timeperiod}.h5")
        Tracks = pd.read_hdf(file, key='table') 
        ds[model][timeperiod] = Tracks

In [4]:
ds["ICON"]["JUNE"].columns

Index(['frame', 'idx_cell', 'hdim_1', 'hdim_2', 'num_cell',
       'threshold_value_cell', 'feature', 'time', 'timestr', 'latitude',
       'longitude', 'ncells_cell', 'mean_precip_cell', 'total_precip_cell',
       'max_precip_cell', 'area_cell', 'd_eq_cell', 'idx_anvil',
       'hdim_1_anvil', 'hdim_2_anvil', 'num_anvil', 'threshold_value_anvil',
       'feature_anvil', 'latitude_anvil', 'longitude_anvil', 'ncells_anvil',
       'mean_iwp', 'total_iwp', 'max_iwp', 'min_tb', 'area_anvil',
       'd_eq_anvil', 'mean_precip_anvil', 'total_precip_anvil',
       'max_precip_anvil', 'cell', 'time_cell', 'v', 'track', 'time_track',
       'rain_amount'],
      dtype='object')

# calculate domain area (nneded for 53 for rain amount of cells) 

In [5]:
# set date
date = "2006-06-06-1800"

# load data
precip = ob.load_clean_crop_imerg(date,30)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

IMERG data loaded for 2006-06-06-1800, cleaned, cropped, 'precipitation' selected, longitude/latitude rename, assigned attrs,drop timedim


In [6]:
def calc_domain_area(da, R=6371000):
    """
    Total domain area in m², summed over all grid cells.
    """
    dlat = np.deg2rad(np.abs(da.latitude.diff("latitude").mean())) # 0.1 
    dlon = np.deg2rad(np.abs(da.longitude.diff("longitude").mean())) # 0.1 
    
    cos_lat = np.cos(np.deg2rad(da.latitude))
    
    # area of a lat/lon grid cell: R² * dlat * dlon * cos(lat)
    area_2d = (R**2) * dlat * dlon * cos_lat * xr.ones_like(da.longitude)

    return area_2d.sum().item()

In [7]:
domainarea = calc_domain_area(precip)
domainarea
# double checked this by calc  => is okay
dxy = 11119.492664455873
tmp = dxy*3600*dxy*600
dif = (domainarea - tmp)
rel_dif = 100*(dif/domainarea)
print(f"{rel_dif:.2f} %")  # => estimated area as with the tobac grid is a bit smaller than the excat grid

-4.72 %


# anvil area covered per timestamp
1. mean anvil area covered
2. calc fraction of mean anvil area covered / domain area

In [8]:
print("mean anvil area covered per timestamp relative to domain area")
for model in models: 
    for timeperiod in timeperiods: 
        anvil_unique = ds[model][timeperiod].drop_duplicates(subset=["time", "feature_anvil"])
        anvilarea_per_time = anvil_unique.groupby("time")["area_anvil"].sum()
        fraction = anvilarea_per_time / (domainarea)
        meanfraction = fraction.mean()*100

        
        print(f"{timeperiod} {model}: {meanfraction:.3g}\,\%")

mean anvil area covered per timestamp relative to domain area
JUNE ICON: 1.74\,\%
DECEMBER ICON: 1.72\,\%
JUNE OBS: 1.21\,\%
DECEMBER OBS: 1.29\,\%


# precipitation captured

In [9]:
print("total rain amount, accumulated across all cells,  normalized by the domain area")
for model in models: 
    for timeperiod in timeperiods: 
        rain =  ds[model][timeperiod]["rain_amount"].sum() / domainarea 
        print(f"{timeperiod} {model}: {rain:.3g}\,mm\,/\,month")

total rain amount, accumulated across all cells,  normalized by the domain area
JUNE ICON: 8.33\,mm\,/\,month
DECEMBER ICON: 8.71\,mm\,/\,month
JUNE OBS: 5.58\,mm\,/\,month
DECEMBER OBS: 6.07\,mm\,/\,month


In [10]:
# load domain-wide precip/month for ICON and OBS, structured as domain_rain[model][timeperiod]
inputdir = Path("../output/aux")
domain_rain = {}
for model in models:
    file = inputdir / f"{model}_monthly_rain_amount.nc"
    file_ds = xr.open_dataset(file, engine="h5netcdf")
    
    #print(file_ds.attrs)
    
    domain_rain[model] = {
        timeperiod: file_ds["rain_amount"].sel(timeperiod=timeperiod).item()
        for timeperiod in timeperiods
    }
    print(f"{model}, total rain amount of month")
    print(f'JUNE:     {domain_rain[model]["JUNE"]:.3g}\,mm\,/\,month')
    print(f'DECEMBER: {domain_rain[model]["DECEMBER"]:.3g}\,mm\,/\,month')

ICON, total rain amount of month
JUNE:     115\,mm\,/\,month
DECEMBER: 127\,mm\,/\,month
OBS, total rain amount of month
JUNE:     96.6\,mm\,/\,month
DECEMBER: 106\,mm\,/\,month


In [11]:

for model in models:
    print(f"\n{model}\ntotal rain amount, accumulated across all cells, normalized by the domain area")
    for timeperiod in timeperiods:
        rain_cells = ds[model][timeperiod]["rain_amount"].sum() / domainarea
        rain_domain = domain_rain[model][timeperiod]
        frac = 100 * rain_cells / rain_domain
        print(f"{timeperiod} {model}: cells={rain_cells:.3g}\,mm\,/\,month | domain={rain_domain:.3g}\,mm\,/\,month | {frac:.1f}\,\% captured")


ICON
total rain amount, accumulated across all cells, normalized by the domain area
JUNE ICON: cells=8.33\,mm\,/\,month | domain=115\,mm\,/\,month | 7.3\,\% captured
DECEMBER ICON: cells=8.71\,mm\,/\,month | domain=127\,mm\,/\,month | 6.8\,\% captured

OBS
total rain amount, accumulated across all cells, normalized by the domain area
JUNE OBS: cells=5.58\,mm\,/\,month | domain=96.6\,mm\,/\,month | 5.8\,\% captured
DECEMBER OBS: cells=6.07\,mm\,/\,month | domain=106\,mm\,/\,month | 5.7\,\% captured


In [12]:

for model in models:
    print(f"\n{model}\ntotal rain amount of all anvils, accumulated over all anvils, normalized by the domain area")
    for timeperiod in timeperiods:
        
        ds[model][timeperiod]["rain_amount_anvil"]= 3 * ds[model][timeperiod]["mean_precip_anvil"] * ds[model][timeperiod]["area_anvil"]
        rain_cells = ds[model][timeperiod]["rain_amount_anvil"].sum() / domainarea
        rain_domain = domain_rain[model][timeperiod]
        frac = 100 * rain_cells / rain_domain
        print(f"{timeperiod} {model}: anvils={rain_cells:.3g}\,mm\,/\,month | domain={rain_domain:.3g}\,mm\,/\,month | {frac:.1f}\,\% captured")


ICON
total rain amount of all anvils, accumulated over all anvils, normalized by the domain area
JUNE ICON: anvils=66.8\,mm\,/\,month | domain=115\,mm\,/\,month | 58.3\,\% captured
DECEMBER ICON: anvils=70.6\,mm\,/\,month | domain=127\,mm\,/\,month | 55.5\,\% captured

OBS
total rain amount of all anvils, accumulated over all anvils, normalized by the domain area
JUNE OBS: anvils=46.9\,mm\,/\,month | domain=96.6\,mm\,/\,month | 48.5\,\% captured
DECEMBER OBS: anvils=61.4\,mm\,/\,month | domain=106\,mm\,/\,month | 58.0\,\% captured


# cell and anvil counts per timestamp

In [16]:
print("mean cell count per timestamp")
for model in models:
    for timeperiod in timeperiods:
        cell_unique = ds[model][timeperiod].drop_duplicates(subset=["time", "feature"])
        ncells_per_time = cell_unique.groupby("time")["feature"].nunique()
        meanncells = ncells_per_time.mean()

        print(f"{timeperiod} {model}: {meanncells:.3g} cells/timestamp")


print("total cell count per month")
for model in models:
    for timeperiod in timeperiods:
        cell_unique = ds[model][timeperiod].drop_duplicates(subset=["time", "feature"])
        ncells_per_time = cell_unique.groupby("time")["feature"].nunique()
        meanncells = ncells_per_time.sum()

        print(f"{timeperiod} {model}: {meanncells:.3g} cells/timestamp")

mean cell count per timestamp
JUNE ICON: 129 cells/timestamp
DECEMBER ICON: 129 cells/timestamp
JUNE OBS: 28.9 cells/timestamp
DECEMBER OBS: 28.5 cells/timestamp
total cell count per month
JUNE ICON: 3.07e+04 cells/timestamp
DECEMBER ICON: 3.18e+04 cells/timestamp
JUNE OBS: 6.85e+03 cells/timestamp
DECEMBER OBS: 7.01e+03 cells/timestamp


In [14]:
print("mean anvil feature count per timestamp")
for model in models:
    for timeperiod in timeperiods:
        anvil_unique = ds[model][timeperiod].drop_duplicates(subset=["time", "feature_anvil"])
        nanvils_per_time = anvil_unique.groupby("time")["feature_anvil"].nunique()
        meannanvils = nanvils_per_time.mean()

        print(f"{timeperiod} {model}: {meannanvils:.3g} anvils/timestamp")

mean anvil feature count per timestamp
JUNE ICON: 74.7 anvils/timestamp
DECEMBER ICON: 77.1 anvils/timestamp
JUNE OBS: 20.3 anvils/timestamp
DECEMBER OBS: 19.2 anvils/timestamp
